In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [9]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [10]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

# opt = torch_numopt.ConjugateGradient(model=model, lr=1e-4, line_search_method="const", cg_method="PR")
opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="armijo", cg_method="PR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="wolfe", cg_method="PR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="strong-wolfe", cg_method="PR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="goldstein", cg_method="PR")


model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.40348929166793823
epoch:  1, loss: 0.26795557141304016
epoch:  2, loss: 0.18090561032295227
epoch:  3, loss: 0.1246228739619255
epoch:  4, loss: 0.08987288177013397
epoch:  5, loss: 0.06882783770561218
epoch:  6, loss: 0.05613606423139572
epoch:  7, loss: 0.0485183522105217
epoch:  8, loss: 0.043967340141534805
epoch:  9, loss: 0.04125833511352539
epoch:  10, loss: 0.03965034335851669
epoch:  11, loss: 0.03869834542274475
epoch:  12, loss: 0.038135506212711334
epoch:  13, loss: 0.037802763283252716
epoch:  14, loss: 0.037605468183755875
epoch:  15, loss: 0.037487976253032684
epoch:  16, loss: 0.03741731494665146
epoch:  17, loss: 0.03737422823905945
epoch:  18, loss: 0.037347402423620224
epoch:  19, loss: 0.03733489289879799
epoch:  20, loss: 0.03730642795562744
epoch:  21, loss: 0.03729575499892235
epoch:  22, loss: 0.0372651033103466
epoch:  23, loss: 0.037251945585012436
epoch:  24, loss: 0.03721913695335388
epoch:  25, loss: 0.03720996156334877
epoch:  26, loss: 

In [11]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7815502279924336
Test metrics:  R2 = 0.7259086615678321


In [12]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

# opt = torch_numopt.ConjugateGradient(model=model, lr=1e-4, line_search_method="const", cg_method="FR")
opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="armijo", cg_method="FR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="wolfe", cg_method="FR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="strong-wolfe", cg_method="FR")
# opt = torch_numopt.ConjugateGradient(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="goldstein", cg_method="FR")

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.20686501264572144
epoch:  1, loss: 0.12932315468788147
epoch:  2, loss: 0.08717010170221329
epoch:  3, loss: 0.06407549977302551
epoch:  4, loss: 0.051455363631248474
epoch:  5, loss: 0.04459924250841141
epoch:  6, loss: 0.04089607298374176
epoch:  7, loss: 0.038901202380657196
epoch:  8, loss: 0.03783875331282616
epoch:  9, loss: 0.03726278617978096
epoch:  10, loss: 0.0369386151432991
epoch:  11, loss: 0.036752283573150635
epoch:  12, loss: 0.03663993626832962
epoch:  13, loss: 0.036567557603120804
epoch:  14, loss: 0.03652695193886757
epoch:  15, loss: 0.03642599657177925
epoch:  16, loss: 0.036368511617183685
epoch:  17, loss: 0.03634997829794884
epoch:  18, loss: 0.03625589609146118
epoch:  19, loss: 0.0362013541162014
epoch:  20, loss: 0.03616642579436302
epoch:  21, loss: 0.03607761487364769
epoch:  22, loss: 0.03602584823966026
epoch:  23, loss: 0.03598729893565178
epoch:  24, loss: 0.035901542752981186
epoch:  25, loss: 0.03585139289498329
epoch:  26, loss: 

In [13]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7614691734465167
Test metrics:  R2 = 0.6995520417279617
